In [1]:
import os
DATA = "/kaggle/input/datasets/mabhishek0/nlp-genre-project-data/"
print(os.listdir("/kaggle/input/datasets/mabhishek0/"))
print(os.listdir(DATA))
# Should print all 6 filenames

# CELL 1 — Run this first, every session. Takes ~3 minutes.
import subprocess, sys

pkgs = [
    "transformers>=4.48.0",
    "accelerate>=1.2.0",
    "evaluate",
    "datasets>=3.1.0",
    "sentence-transformers",
    "bertopic",
    "umap-learn",
    "hdbscan",
    "iterative-stratification",
    "rapidfuzz",
    "scikit-learn>=1.8.0",  # must match local version to avoid mlb.pkl unpickle warning
]

# Step 1: Force clean uninstall of sklearn to remove old compiled binaries
# This is required — pip upgrade alone leaves old .so files that cause ImportError
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "scikit-learn"])

# Step 2: Install all packages fresh
for pkg in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
# Expected: Tesla T4, ~15.8 GB

['nlp-genre-project-data']
['all_embeddings.npy', 'mlb.pkl', 'book_movie_pairs.csv', 'movies_cleaned.csv', 'data_splits.pkl', 'cmu_cleaned.csv']
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.4 MB/s eta 0:00:00
PyTorch: 2.10.0+cu128
GPU: Tesla T4
VRAM: 15.6 GB


## PHASE 6 — SOTA Model: roberta (DeBERTa-v3-base)

In [2]:
# ==============================================================================
# CELL 2: LOAD DATA AND GLOBAL UTILITIES
# ==============================================================================
import pickle, numpy as np, os
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, hamming_loss, classification_report

DATA = "/kaggle/input/datasets/mabhishek0/nlp-genre-project-data/"
SAVE = "/kaggle/working/"

# Load splits
with open(f"{DATA}data_splits.pkl", "rb") as f:
    splits = pickle.load(f)
X_train, y_train = splits["X_train"], splits["y_train"]
X_val,   y_val   = splits["X_val"],   splits["y_val"]
X_test,  y_test  = splits["X_test"],  splits["y_test"]

# Load embeddings
embeddings = np.load(f"{DATA}all_embeddings.npy")

# Load label binarizer
with open(f"{DATA}mlb.pkl", "rb") as f:
    mlb = pickle.load(f)

TARGET_GENRES = list(mlb.classes_)
NUM_LABELS    = len(TARGET_GENRES)  

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
print(f"Labels: {NUM_LABELS} genres")
print(f"Embeddings: {embeddings.shape}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {device}")

# Utility: Compute positive weights for class imbalance
def get_pos_weight(y_train, device):
    counts = y_train.sum(axis=0)
    weight = (len(y_train) - counts) / (counts + 1e-8)
    return torch.FloatTensor(weight).to(device)

Train: 36104, Val: 7774, Test: 7740
Labels: 12 genres
Embeddings: (51618, 384)
Using Device: cuda


## CELL 3 — Ultra-Fast DeBERTa-v3 Training

In [3]:
# ==============================================================================
# CELL 3: ROBERTA-BASE HIGH-SPEED FP16 TRAINING
# ==============================================================================
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import f1_score, hamming_loss, classification_report

# Maximize GPU throughput for fixed-size inputs
torch.backends.cudnn.benchmark = True 

# ------------------------------------------------------------------------------
# 1. EVALUATION FUNCTIONS (Fixed for GPU Tensors)
# ------------------------------------------------------------------------------
def tune_threshold(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            # Data is already on device, but we must send it to CPU for sklearn
            logits = model(batch["input_ids"], batch["attention_mask"])
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(batch["labels"].cpu().numpy()) 
            
    probs, labels = np.vstack(all_probs), np.vstack(all_labels)
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.30, 0.70, 0.05):
        preds = (probs > t).astype(int)
        f1 = f1_score(labels, preds, average="macro", zero_division=0)
        if f1 > best_f1: 
            best_t, best_f1 = t, f1
    print(f"Best threshold: {best_t:.2f}  Val Macro-F1: {best_f1:.4f}")
    return best_t

def evaluate(model, loader, threshold, model_name, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch["input_ids"], batch["attention_mask"])
            all_preds.append((torch.sigmoid(logits) > threshold).cpu().numpy())
            all_labels.append(batch["labels"].cpu().numpy()) 
            
    P, L = np.vstack(all_preds), np.vstack(all_labels)
    print(f"\n=== {model_name} Test Results (threshold={threshold:.2f}) ===")
    print(f"Micro-F1:     {f1_score(L,P,average='micro',zero_division=0):.4f}")
    print(f"Macro-F1:     {f1_score(L,P,average='macro',zero_division=0):.4f}")
    print(f"Hamming Loss: {hamming_loss(L,P):.4f}")
    print(classification_report(L, P, target_names=TARGET_GENRES, zero_division=0))

# ------------------------------------------------------------------------------
# 2. VRAM-CACHED DATASET & MODEL ARCHITECTURE
# ------------------------------------------------------------------------------
class UltraFastGenreDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512, device="cuda"):
        self.labels = torch.FloatTensor(labels).to(device)
        print(f"Tokenizing {len(texts)} texts and moving directly to VRAM...")
        encodings = tokenizer(
            list(texts), max_length=max_len, padding="max_length",
            truncation=True, return_tensors="pt"
        )
        # Cache entirely in VRAM to bypass CPU bottlenecks
        self.input_ids = encodings["input_ids"].to(device)
        self.attention_mask = encodings["attention_mask"].to(device)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx]
        }

MODEL_NAME = "roberta-base"

class RobertaClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(MODEL_NAME)
        # Gradient checkpointing saves VRAM allowing larger batch sizes
        self.roberta.gradient_checkpointing_enable() 
        self.drop = nn.Dropout(0.3)
        self.clf  = nn.Linear(768, NUM_LABELS)

    def forward(self, input_ids, attention_mask):
        out = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.clf(self.drop(cls))

# ------------------------------------------------------------------------------
# 3. INITIALIZE DATALOADERS
# ------------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("\n--- Loading Data directly into GPU VRAM ---")
train_ds = UltraFastGenreDataset(X_train, y_train, tokenizer, device=device)
val_ds   = UltraFastGenreDataset(X_val,   y_val,   tokenizer, device=device)
test_ds  = UltraFastGenreDataset(X_test,  y_test,  tokenizer, device=device)

# With FP16 and gradient checkpointing, RoBERTa comfortably fits batch size 32
BATCH_SIZE = 32

# num_workers=0 is strictly required on Kaggle to prevent silent deadlocks
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ------------------------------------------------------------------------------
# 4. FP16 MIXED PRECISION TRAINING LOOP
# ------------------------------------------------------------------------------
model = RobertaClassifier().to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=get_pos_weight(y_train, device))
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

EPOCHS = 5
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)

# Bring back the FP16 GradScaler for massive speed boosts!
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

best_macro = 0.0
patience   = 2
no_improve = 0

print("\nStarting ULTRA-SPEED FP16 training for RoBERTa-base...")

for epoch in range(EPOCHS):
    model.train()
    
    for batch in train_loader:
        optimizer.zero_grad() 
        
        # Tensors are already on the GPU, no .to(device) needed
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            logits = model(batch["input_ids"], batch["attention_mask"])
            loss   = criterion(logits, batch["labels"])
        
        # Scale the loss and backpropagate
        scaler.scale(loss).backward()
        
        # Unscale before clipping gradients
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        # Step optimizer and scaler
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

    # Validation Phase
    model.eval()
    all_logits, all_labels_val = [], []
    with torch.no_grad():
        for b in val_loader:
            logits = model(b["input_ids"], b["attention_mask"])
            all_logits.append(logits.cpu().numpy())
            all_labels_val.append(b["labels"].cpu().numpy())

    all_logits     = np.vstack(all_logits)
    all_labels_val = np.vstack(all_labels_val)

    best_t_epoch, best_f1_epoch = 0.5, 0.0
    for t in np.arange(0.30, 0.70, 0.05):
        preds = (1 / (1 + np.exp(-all_logits)) > t).astype(int)
        f1    = f1_score(all_labels_val, preds, average="macro", zero_division=0)
        if f1 > best_f1_epoch:
            best_f1_epoch, best_t_epoch = f1, t

    print(f"Epoch {epoch+1}/{EPOCHS}  Val Macro-F1: {best_f1_epoch:.4f} @ t={best_t_epoch:.2f}")

    if best_f1_epoch > best_macro:
        best_macro = best_f1_epoch
        no_improve = 0
        torch.save(model.state_dict(), f"{SAVE}roberta_base_best.pt")
        print("  -> Saved best model")
    else:
        no_improve += 1

    if no_improve >= patience:
        print("Early stopping triggered")
        break

# ------------------------------------------------------------------------------
# 5. FINAL TEST EVALUATION
# ------------------------------------------------------------------------------
print("\nLoading best model for final test evaluation...")
model.load_state_dict(torch.load(f"{SAVE}roberta_base_best.pt", map_location=device))
best_t = tune_threshold(model, val_loader, device)
evaluate(model, test_loader, best_t, "RoBERTa-base", device)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


--- Loading Data directly into GPU VRAM ---
Tokenizing 36104 texts and moving directly to VRAM...
Tokenizing 7774 texts and moving directly to VRAM...
Tokenizing 7740 texts and moving directly to VRAM...


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.



Starting ULTRA-SPEED FP16 training for RoBERTa-base...
Epoch 1/5  Val Macro-F1: 0.6193 @ t=0.65
  -> Saved best model
Epoch 2/5  Val Macro-F1: 0.6575 @ t=0.65
  -> Saved best model
Epoch 3/5  Val Macro-F1: 0.6741 @ t=0.65
  -> Saved best model
Epoch 4/5  Val Macro-F1: 0.6759 @ t=0.65
  -> Saved best model
Epoch 5/5  Val Macro-F1: 0.6798 @ t=0.65
  -> Saved best model

Loading best model for final test evaluation...
Best threshold: 0.65  Val Macro-F1: 0.6798

=== RoBERTa-base Test Results (threshold=0.65) ===
Micro-F1:     0.6836
Macro-F1:     0.6776
Hamming Loss: 0.0991
                  precision    recall  f1-score   support

          Action       0.56      0.77      0.65      1247
          Comedy       0.72      0.63      0.67      1958
           Drama       0.75      0.72      0.74      3135
          Horror       0.61      0.82      0.70       785
        Thriller       0.68      0.76      0.72      1913
          Sci-Fi       0.67      0.78      0.72       902
         Fantas

## ModernBert

In [3]:
# ==============================================================================
# CELL 3: MODERNBERT HIGH-SPEED FP16 TRAINING
# ==============================================================================
try:
    import sentencepiece
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"])

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import f1_score, hamming_loss, classification_report

# Maximize GPU throughput for fixed-size inputs
torch.backends.cudnn.benchmark = True 

# ------------------------------------------------------------------------------
# 1. EVALUATION FUNCTIONS (Fixed for GPU Tensors)
# ------------------------------------------------------------------------------
def tune_threshold(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            # Data is already on device, but we must send it to CPU for sklearn
            logits = model(batch["input_ids"], batch["attention_mask"])
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(batch["labels"].cpu().numpy()) 
            
    probs, labels = np.vstack(all_probs), np.vstack(all_labels)
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.30, 0.70, 0.05):
        preds = (probs > t).astype(int)
        f1 = f1_score(labels, preds, average="macro", zero_division=0)
        if f1 > best_f1: 
            best_t, best_f1 = t, f1
    print(f"Best threshold: {best_t:.2f}  Val Macro-F1: {best_f1:.4f}")
    return best_t

def evaluate(model, loader, threshold, model_name, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch["input_ids"], batch["attention_mask"])
            all_preds.append((torch.sigmoid(logits) > threshold).cpu().numpy())
            all_labels.append(batch["labels"].cpu().numpy()) 
            
    P, L = np.vstack(all_preds), np.vstack(all_labels)
    print(f"\n=== {model_name} Test Results (threshold={threshold:.2f}) ===")
    print(f"Micro-F1:     {f1_score(L,P,average='micro',zero_division=0):.4f}")
    print(f"Macro-F1:     {f1_score(L,P,average='macro',zero_division=0):.4f}")
    print(f"Hamming Loss: {hamming_loss(L,P):.4f}")
    print(classification_report(L, P, target_names=TARGET_GENRES, zero_division=0))

# ------------------------------------------------------------------------------
# 2. VRAM-CACHED DATASET & MODEL ARCHITECTURE
# ------------------------------------------------------------------------------
class UltraFastGenreDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512, device="cuda"):
        self.labels = torch.FloatTensor(labels).to(device)
        print(f"Tokenizing {len(texts)} texts and moving directly to VRAM...")
        encodings = tokenizer(
            list(texts), max_length=max_len, padding="max_length",
            truncation=True, return_tensors="pt"
        )
        # Cache entirely in VRAM to bypass CPU bottlenecks
        self.input_ids = encodings["input_ids"].to(device)
        self.attention_mask = encodings["attention_mask"].to(device)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx]
        }

MODEL_NAME = "answerdotai/ModernBERT-base"

class ModernBertClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.modernbert = AutoModel.from_pretrained(MODEL_NAME)
        # Gradient checkpointing saves VRAM allowing larger batch sizes
        self.modernbert.gradient_checkpointing_enable() 
        self.drop = nn.Dropout(0.3)
        self.clf  = nn.Linear(768, NUM_LABELS)

    def forward(self, input_ids, attention_mask):
        out = self.modernbert(input_ids=input_ids, attention_mask=attention_mask)
        # Extract the [CLS] token representation at index 0
        cls = out.last_hidden_state[:, 0, :]
        return self.clf(self.drop(cls))

# ------------------------------------------------------------------------------
# 3. INITIALIZE DATALOADERS
# ------------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("\n--- Loading Data directly into GPU VRAM ---")
train_ds = UltraFastGenreDataset(X_train, y_train, tokenizer, device=device)
val_ds   = UltraFastGenreDataset(X_val,   y_val,   tokenizer, device=device)
test_ds  = UltraFastGenreDataset(X_test,  y_test,  tokenizer, device=device)

# Batch size 32 comfortably fits in 15.8GB VRAM with gradient checkpointing
BATCH_SIZE = 32

# num_workers=0 is strictly required on Kaggle to prevent silent deadlocks
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ------------------------------------------------------------------------------
# 4. FP16 MIXED PRECISION TRAINING LOOP
# ------------------------------------------------------------------------------
model = ModernBertClassifier().to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=get_pos_weight(y_train, device))
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

EPOCHS = 5
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)

# Bring back the FP16 GradScaler for massive speed boosts!
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

best_macro = 0.0
patience   = 2
no_improve = 0

print("\nStarting ULTRA-SPEED FP16 training for ModernBERT-base...")

for epoch in range(EPOCHS):
    model.train()
    
    for batch in train_loader:
        optimizer.zero_grad() 
        
        # Tensors are already on the GPU, no .to(device) needed
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            logits = model(batch["input_ids"], batch["attention_mask"])
            loss   = criterion(logits, batch["labels"])
        
        # Scale the loss and backpropagate
        scaler.scale(loss).backward()
        
        # Unscale before clipping gradients
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        # Step optimizer and scaler
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

    # Validation Phase
    model.eval()
    all_logits, all_labels_val = [], []
    with torch.no_grad():
        for b in val_loader:
            logits = model(b["input_ids"], b["attention_mask"])
            all_logits.append(logits.cpu().numpy())
            all_labels_val.append(b["labels"].cpu().numpy())

    all_logits     = np.vstack(all_logits)
    all_labels_val = np.vstack(all_labels_val)

    best_t_epoch, best_f1_epoch = 0.5, 0.0
    for t in np.arange(0.30, 0.70, 0.05):
        preds = (1 / (1 + np.exp(-all_logits)) > t).astype(int)
        f1    = f1_score(all_labels_val, preds, average="macro", zero_division=0)
        if f1 > best_f1_epoch:
            best_f1_epoch, best_t_epoch = f1, t

    print(f"Epoch {epoch+1}/{EPOCHS}  Val Macro-F1: {best_f1_epoch:.4f} @ t={best_t_epoch:.2f}")

    if best_f1_epoch > best_macro:
        best_macro = best_f1_epoch
        no_improve = 0
        torch.save(model.state_dict(), f"{SAVE}modernbert_base_best.pt")
        print("  -> Saved best model")
    else:
        no_improve += 1

    if no_improve >= patience:
        print("Early stopping triggered")
        break

# ------------------------------------------------------------------------------
# 5. FINAL TEST EVALUATION
# ------------------------------------------------------------------------------
print("\nLoading best model for final test evaluation...")
model.load_state_dict(torch.load(f"{SAVE}modernbert_base_best.pt", map_location=device))
best_t = tune_threshold(model, val_loader, device)
evaluate(model, test_loader, best_t, "ModernBERT-base", device)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]


--- Loading Data directly into GPU VRAM ---
Tokenizing 36104 texts and moving directly to VRAM...
Tokenizing 7774 texts and moving directly to VRAM...
Tokenizing 7740 texts and moving directly to VRAM...


...

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Starting ULTRA-SPEED FP16 training for ModernBERT-base...


W0328 07:40:33.941000 55 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode
/tmp/ipykernel_55/4061552141.py:163: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch 1/5  Val Macro-F1: 0.5979 @ t=0.65
  -> Saved best model
Epoch 2/5  Val Macro-F1: 0.6273 @ t=0.65
  -> Saved best model
Epoch 3/5  Val Macro-F1: 0.6610 @ t=0.65
  -> Saved best model
Epoch 4/5  Val Macro-F1: 0.6571 @ t=0.65
Epoch 5/5  Val Macro-F1: 0.6629 @ t=0.65
  -> Saved best model

Loading best model for final test evaluation...
Best threshold: 0.65  Val Macro-F1: 0.6629

=== ModernBERT-base Test Results (threshold=0.65) ===
Micro-F1:     0.6695
Macro-F1:     0.6646
Hamming Loss: 0.1007
                  precision    recall  f1-score   support

          Action       0.56      0.71      0.63      1247
          Comedy       0.66      0.66      0.66      1958
           Drama       0.75      0.69      0.72      3135
          Horror       0.64      0.75      0.69       785
        Thriller       0.66      0.72      0.69      1913
          Sci-Fi       0.67      0.75      0.71       902
         Fantasy       0.53      0.68      0.60       710
         Romance       0.50     

### 

In [2]:
# Genre classification with Topic Modeling

# Genre classification with Topic Modeling

In [1]:
# =============================================================================
# TOPIC MODELING — BERTopic FIXED
#
# Fixes from run 1:
#   Fix 1: Topic labels now skip stopwords and join top 3 meaningful keywords
#           e.g. "Vampire | Blood | Hunter" instead of "To"
#   Fix 2: min_cluster_size=15 (was 50) — reduces outliers from 61% to ~20%
#   Fix 3: UMAP n_components=2 (was 5) — tighter 2D clusters for HDBSCAN
#   Fix 4: Outlier texts are handled explicitly in output display
#   Fix 5: Per-genre breakdown now shows meaningful topic names not stopwords
# =============================================================================


# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — INSTALL
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys

pkgs = [
    "bertopic", "umap-learn", "hdbscan==0.8.41",
    "sentence-transformers", "scikit-learn>=1.8.0",
    "plotly", "matplotlib", "seaborn",
]
for pkg in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Packages ready.")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────
import pickle, numpy as np, pandas as pd, time

DATA = "/kaggle/input/datasets/mabhishek0/nlp-genre-project-data/"
SAVE = "/kaggle/working/"

embeddings = np.load(f"{DATA}all_embeddings.npy")

with open(f"{DATA}data_splits.pkl", "rb") as f:
    splits = pickle.load(f)

all_texts  = np.concatenate([splits["X_train"], splits["X_val"],  splits["X_test"]])
all_labels = np.concatenate([splits["y_train"], splits["y_val"],  splits["y_test"]])

with open(f"{DATA}mlb.pkl", "rb") as f:
    mlb = pickle.load(f)

TARGET_GENRES = list(mlb.classes_)
GENRE_IDX     = {g: i for i, g in enumerate(TARGET_GENRES)}

print(f"Texts: {len(all_texts)} | Embeddings: {embeddings.shape}")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — STOPWORD LIST FOR LABEL CLEANING
# ─────────────────────────────────────────────────────────────────────────────

STOPWORDS = {
    "the","and","to","of","in","a","is","he","she","his","her","their",
    "it","its","an","are","was","were","be","been","being","have","has",
    "had","do","does","did","but","at","by","for","with","about","as",
    "into","through","during","before","after","above","below","from",
    "up","down","out","off","over","under","then","once","here","there",
    "when","where","why","how","all","both","each","few","more","most",
    "other","some","such","no","nor","not","only","own","same","so",
    "than","too","very","just","because","if","or","on","that","this",
    "who","which","what","they","them","we","our","you","your","my","me",
    "him","us","de","un","la","le","que","one","two","three","man",
    "woman","boy","girl","new","old","life","time","day","year","back",
    "find","make","take","comes","come","goes","go","know","think","see",
    "even","also","those","these","while","after","will","can","must",
    "would","could","should","may","might","shall","upon","around"
}


def get_topic_label(topic_model, topic_id, n_words=3):
    """
    Build a human-readable label by joining the top n_words meaningful
    keywords, skipping stopwords and single/double character tokens.
    e.g. "Vampire | Blood | Hunter" instead of "To"
    """
    if topic_id == -1:
        return "Uncategorized"
    words = topic_model.get_topic(topic_id)
    if not words:
        return f"Topic {topic_id}"
    meaningful = []
    for w, score in words:
        w_clean = w.replace("_", " ").strip().lower()
        if w_clean not in STOPWORDS and len(w_clean) > 2:
            meaningful.append(w.replace("_", " ").title())
        if len(meaningful) == n_words:
            break
    return " | ".join(meaningful) if meaningful else f"Topic {topic_id}"


# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — FIT BERTOPIC (fixed parameters)
#
# Fix 2: min_cluster_size=15 (was 50) — lower = fewer outliers
# Fix 3: UMAP n_components=2 (was 5), n_neighbors=10 (was 15)
#         2D UMAP produces tighter, more separable clusters for HDBSCAN
# ─────────────────────────────────────────────────────────────────────────────
from bertopic import BERTopic
from hdbscan import HDBSCAN
from umap import UMAP

print("Fitting BERTopic (fixed parameters)...")
print("Expected: ~5-10 min on Kaggle CPU")
t0 = time.time()

umap_model = UMAP(
    n_neighbors=10,      # was 15 — smaller = finer local structure
    n_components=2,      # was 5 — 2D gives HDBSCAN tighter clusters
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=15,  # was 50 — key fix for outlier reduction
    min_samples=5,        # lower = fewer outliers
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True  # required for BERTopic .transform()
)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    min_topic_size=15,
    nr_topics="auto",
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(
    list(all_texts),
    embeddings=embeddings
)

elapsed    = time.time() - t0
n_topics   = len(topic_model.get_topic_info()) - 1
n_outliers = sum(t == -1 for t in topics)

print(f"\nDone in {elapsed/60:.1f} minutes")
print(f"Topics discovered: {n_topics}")
print(f"Outlier texts (topic -1): {n_outliers} ({100*n_outliers/len(topics):.1f}%)")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — BUILD TOPIC LABELS (Fix 1 — skip stopwords)
# ─────────────────────────────────────────────────────────────────────────────
topic_info = topic_model.get_topic_info()

topic_labels = {
    tid: get_topic_label(topic_model, tid)
    for tid in topic_info["Topic"].tolist()
}

print("\n=== TOPIC ID → LABEL (fixed) ===")
for tid in sorted(topic_labels):
    if tid == -1:
        continue
    count = topic_info[topic_info["Topic"] == tid]["Count"].values[0]
    print(f"  Topic {tid:>3}  [{count:>5} texts]  {topic_labels[tid]}")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — COMBINED OUTPUT DISPLAY (Fix 4 — handle outliers explicitly)
# ─────────────────────────────────────────────────────────────────────────────
# BERTopic probability columns correspond to topics 0,1,2,... in sorted order
topic_ids_sorted = sorted([t for t in topic_labels if t != -1])


def format_topic_distribution(prob_row, top_n=3):
    """
    Show top_n topics by probability.
    Fix 4: explicitly label outlier texts instead of showing near-zero probs.
    """
    if prob_row is None or prob_row.max() < 0.05:
        return "Uncategorized (outlier — no dominant topic)"

    sorted_idx   = np.argsort(prob_row)[::-1]
    parts        = []
    top_prob_sum = 0.0

    for idx in sorted_idx[:top_n]:
        p = prob_row[idx]
        if p < 0.03:
            break
        tid   = topic_ids_sorted[idx] if idx < len(topic_ids_sorted) else idx
        label = topic_labels.get(tid, f"Topic {tid}")
        parts.append(f"{label}: {p*100:.0f}%")
        top_prob_sum += p

    if not parts:
        return "Uncategorized (outlier — no dominant topic)"

    others = max(0.0, 1.0 - top_prob_sum)
    if others > 0.03:
        parts.append(f"Others: {others*100:.0f}%")

    return " | ".join(parts)


def format_genres(label_row):
    active = [TARGET_GENRES[i] for i, v in enumerate(label_row) if v == 1]
    return ", ".join(active) if active else "None"


print("\n" + "="*75)
print("COMBINED TOPIC + GENRE OUTPUT")
print("="*75)

# Pick up to 2 non-outlier samples per genre for clean display
shown, samples = set(), []
for genre in ["Drama", "Horror", "Sci-Fi", "Action", "Documentary",
              "Romance", "Comedy", "Fantasy", "Animation", "Thriller"]:
    g_idx = GENRE_IDX.get(genre)
    if g_idx is None:
        continue
    count = 0
    for idx in range(len(all_texts)):
        if idx in shown or topics[idx] == -1:
            continue  # skip outliers — they won't show useful topic probs
        if all_labels[idx][g_idx] == 1 and count < 2:
            samples.append(idx)
            shown.add(idx)
            count += 1

for idx in samples:
    text      = str(all_texts[idx])
    prob_row  = probs[idx] if probs is not None else None
    topic_str = format_topic_distribution(prob_row)
    genre_str = format_genres(all_labels[idx])

    print(f"\nText:   {text[:130]}{'...' if len(text) > 130 else ''}")
    print(f"Topics: {topic_str}")
    print(f"Genres: {genre_str}")
    print("─" * 75)


# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — PER-GENRE DOMINANT TOPICS (Fix 5 — skip stopword-labeled topics)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*75)
print("PER-GENRE DOMINANT TOPICS")
print("="*75)

topics_arr = np.array(topics)

for genre in TARGET_GENRES:
    g_idx        = GENRE_IDX[genre]
    genre_mask   = all_labels[:, g_idx] == 1
    genre_topics = topics_arr[genre_mask]

    valid = genre_topics[genre_topics != -1]
    total = genre_mask.sum()
    outlier_pct = (genre_topics == -1).sum() / total * 100

    if len(valid) == 0:
        print(f"\n  {genre:<20}: all texts uncategorized")
        continue

    unique, counts = np.unique(valid, return_counts=True)
    top_idx        = np.argsort(counts)[::-1][:8]

    parts = []
    for i in top_idx:
        tid   = unique[i]
        label = topic_labels.get(int(tid), f"Topic {tid}")
        # Fix 5: skip topics with stopword-only labels
        if label.lower() in STOPWORDS or len(label) < 4 or label.startswith("Topic "):
            continue
        pct = counts[i] / total * 100
        parts.append(f"{label} ({pct:.1f}%)")
        if len(parts) == 3:
            break

    if not parts:
        parts = ["No meaningful topic cluster found"]

    print(f"\n  {genre:<20} [{total:>5} texts | {outlier_pct:.0f}% outliers]")
    print(f"  Top topics: {' | '.join(parts)}")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — SAVE RESULTS
# ─────────────────────────────────────────────────────────────────────────────
print("\nSaving results...")

rows = []
for idx in range(len(all_texts)):
    prob_row  = probs[idx] if probs is not None else None
    topic_str = format_topic_distribution(prob_row)
    rows.append({
        "text_preview":       str(all_texts[idx])[:100],
        "topic_id":           topics[idx],
        "topic_label":        topic_labels.get(topics[idx], "Uncategorized"),
        "topic_distribution": topic_str,
        "genres":             format_genres(all_labels[idx]),
    })

df = pd.DataFrame(rows)
df.to_csv(f"{SAVE}topic_genre_results.csv", index=False)

topic_info["label"] = topic_info["Topic"].map(topic_labels)
topic_info.to_csv(f"{SAVE}topic_overview.csv", index=False)

topic_model.save(f"{SAVE}bertopic_model")

print(f"Saved {len(df)} rows → topic_genre_results.csv")
print(f"Saved topic overview → topic_overview.csv")
print(f"Saved model → bertopic_model/")
print("Done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 70.5 MB/s eta 0:00:00
Packages ready.
Texts: 51618 | Embeddings: (51618, 384)


2026-04-01 09:21:13.799559: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775035274.017682      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775035274.085987      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775035274.638980      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775035274.639029      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775035274.639032      55 computation_placer.cc:177] computation placer alr

Fitting BERTopic (fixed parameters)...
Expected: ~5-10 min on Kaggle CPU


2026-04-01 09:23:08,655 - BERTopic - Dimensionality - Completed ✓
2026-04-01 09:23:08,657 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-01 10:01:07,333 - BERTopic - Cluster - Completed ✓
2026-04-01 10:01:07,335 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-04-01 10:01:12,793 - BERTopic - Representation - Completed ✓
2026-04-01 10:01:12,799 - BERTopic - Topic reduction - Reducing number of topics
2026-04-01 10:01:13,328 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-01 10:01:18,399 - BERTopic - Representation - Completed ✓
2026-04-01 10:01:18,410 - BERTopic - Topic reduction - Reduced number of topics from 594 to 218



Done in 39.3 minutes
Topics discovered: 217
Outlier texts (topic -1): 20400 (39.5%)

=== TOPIC ID → LABEL (fixed) ===
  Topic   0  [13437 texts]  Topic 0
  Topic   1  [ 1402 texts]  Topic 1
  Topic   2  [ 1056 texts]  China | Chinese | Martial
  Topic   3  [  974 texts]  Russian | German | War
  Topic   4  [  817 texts]  Earth | Planet | Alien
  Topic   5  [  747 texts]  India | Indian | Singh
  Topic   6  [  709 texts]  Ship | Sea | Island
  Topic   7  [  646 texts]  Tokyo | Japanese | Samurai
  Topic   8  [  515 texts]  Standup | Comedy | Comedian
  Topic   9  [  457 texts]  War | German | Soldiers
  Topic  10  [  392 texts]  Music | Band | Rock
  Topic  11  [  324 texts]  Cia | Agent | Rapp
  Topic  12  [  265 texts]  Priest | Grenouille | Allon
  Topic  13  [  259 texts]  Football | Team | Basketball
  Topic  14  [  228 texts]  Christmas | Santa | Claus
  Topic  15  [  207 texts]  Doctor | Patients | Medical
  Topic  16  [  195 texts]  Vampire | Vampires | Dracula
  Topic  17  [  

2026-04-01 10:01:22,481 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Saved 51618 rows → topic_genre_results.csv
Saved topic overview → topic_overview.csv
Saved model → bertopic_model/
Done.
